### Kaczmarz algorithm

Iterative method to find the solution of square linear systems $A \vec{x} = \vec{b}$. The iterative step is defined as:

$$
    \vec{x_{k + 1}} = \vec{x_k} + \frac{b_j - \vec{a_j} \cdot \vec{x_k}}{\vec{a_j} \cdot \vec{a_j}}
$$

Where $\vec{a_j}$ is the j-th **row** of matrix $A$. For each step, the j-th row is choosen randomly with probability proportional to its squared norm

In [1]:
import numpy as np

In [ ]:
def Kaczmarz(A, b, x0 = None, eps = 1e-04, max_iter = 1_000):
    A = np.asarray(A, dtype = float)
    b = np.asarray(b, dtype = float)
    N = A.shape[0]

    x = np.asarray(x0, dtype = float).copy() if x0 is not None else np.zeros(N)

    # Sampling probabilities proportional to |a_i|^2
    row_norms_sq = np.sum(A ** 2, axis = 1)
    p = row_norms_sq / (np.linalg.norm(A, 'fro') ** 2)

    iterations = 0
    residual = b - A @ x
    res_norm = np.linalg.norm(residual)

    for _ in range(max_iter):

        i = np.random.choice(N, p = p)
        a_i = A[i, :]
        x += (b[i] - np.dot(a_i, x)) / row_norms_sq[i] * a_i

        residual = b - A @ x
        res_norm = np.linalg.norm(residual)
        if res_norm < eps: # Early stopping criterion active
            break
        
        iterations += 1

    else: # Loop finished without break
        if res_norm >= eps:
            print("Maximum number of iterations reached")

    stats = {
        "iterations": iterations,
        "residual": residual,
        "residual_norm": res_norm,
    }
    return x, stats

In [3]:
# Linear system A @ x = b
A = np.array([
    [ 2, -3,  5, -2],
    [ 4,  2, -3,  7],
    [-3,  5,  2, -3],
    [ 5, -1, -4,  2]
])

b = np.array([4, -1, 7, -3])

In [4]:
x, stats = Kaczmarz(A, b) # Approximated solution
x, stats

(array([ 0.63453719,  1.12983093,  1.07749417, -0.36647544]),
 {'iterations': 281,
  'residual': array([-3.33344404e-06,  0.00000000e+00,  4.22324666e-05,  7.25601527e-05]),
  'residual_norm': np.float64(8.402183551812132e-05)})

In [5]:
err, err_stats = Kaczmarz(A, b - A @ x, eps = 1e-07) # Error estimation
err, err_stats

(array([ 1.77688611e-05,  1.19923502e-05, -7.26320228e-06, -1.66928216e-05]),
 {'iterations': 176,
  'residual': array([3.62524656e-08, 1.35525272e-20, 2.52385302e-08, 4.10317099e-08]),
  'residual_norm': np.float64(6.028951721066931e-08)})

In [6]:
np.linalg.solve(A, b) # Exact solution

array([ 0.63455497,  1.12984293,  1.07748691, -0.36649215])